In [27]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential
from sklearn.inspection import permutation_importance

frequency = "1d"
window_pred = 7

# Cargar los datos para esta frecuencia
BTCUSDT = pd.read_csv('BTCUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
AVAXUSDT = pd.read_csv('AVAXUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
XRPUSDT = pd.read_csv('XRPUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
data = pd.DataFrame() 

data['BTCUSDT'] = BTCUSDT[['close']]
data['AVAXUSDT'] = AVAXUSDT[['close']]
data['XRPUSDT'] = XRPUSDT[['close']]
data.dropna(inplace=True)
data.head()

,BTCUSDT,AVAXUSDT,XRPUSDT
timestamp,,,
2020-09-22,10529.61,5.3193,0.23302
2020-09-23,10241.46,3.5350,0.22164
2020-09-24,10736.32,4.6411,0.23276
2020-09-25,10686.67,4.7134,0.24154
2020-09-26,10728.60,4.5200,0.24153


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [28]:
def save_results_global(model, crypto, acc, sample, frequency='1d'):
    file_name = f'accuracy_results_{frequency}_charac.csv'

    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])

    new_row = pd.DataFrame([[model, crypto, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'Vald/Test'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    df_results.to_csv(file_name, index=False)

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [29]:
def charact_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    #df['ten'] = np.where(df[ric].shift(window_pred) > df[ric], 1, 0)
    print(df['d'].value_counts(normalize=True)) #comprueba si los datos están desbalanceados 
    features = [ric, 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = charact_lags(data, ric, lags, window_pred)
    dfs[ric] = df.dropna(), cols

d
1    0.535387
0    0.464613
Name: proportion, dtype: float64
d
0    0.515072
1    0.484928
Name: proportion, dtype: float64
d
0    0.52228
1    0.47772
Name: proportion, dtype: float64


Comentar que he mirado si los datos están desbalanceados 

In [30]:
dfs[ric][0]

,XRPUSDT,r,sma,min,max,mom,vol,rsi,atr,d,...,rsi_lag_1,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-10-27,0.25273,0.018490,0.247623,0.23273,0.25709,0.038588,0.018629,57.099688,0.003870,0,...,55.317583,57.840898,59.440599,58.907397,60.127914,0.003843,0.003802,0.003828,0.003908,0.003961
2020-10-28,0.24540,-0.029432,0.247767,0.23273,0.25709,0.017962,0.019369,53.598651,0.003985,0,...,57.099688,55.317583,57.840898,59.440599,58.907397,0.003870,0.003843,0.003802,0.003828,0.003908
2020-10-29,0.24239,-0.012342,0.247752,0.23273,0.25709,-0.001894,0.019465,52.238035,0.003952,1,...,53.598651,57.099688,55.317583,57.840898,59.440599,0.003985,0.003870,0.003843,0.003802,0.003828
2020-10-30,0.23905,-0.013875,0.247663,0.23273,0.25709,-0.011005,0.019612,50.758935,0.003932,1,...,52.238035,53.598651,57.099688,55.317583,57.840898,0.003952,0.003985,0.003870,0.003843,0.003802
2020-10-31,0.23968,0.002632,0.247713,0.23273,0.25709,0.006256,0.019431,51.029495,0.003822,1,...,50.758935,52.238035,53.598651,57.099688,55.317583,0.003932,0.003952,0.003985,0.003870,0.003843
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-21,2.23800,-0.018286,2.147353,1.39840,2.72450,0.794707,0.078791,61.230947,0.087947,0,...,62.204662,61.629097,63.277124,69.778755,68.774720,0.089555,0.091254,0.091942,0.086250,0.086355
2024-12-22,2.20310,-0.015717,2.171703,1.39840,2.72450,0.496061,0.073957,60.404383,0.086179,0,...,61.230947,62.204662,61.629097,63.277124,69.778755,0.087947,0.089555,0.091254,0.091942,0.086250
2024-12-23,2.26170,0.026251,2.198143,1.39840,2.72450,0.540143,0.073927,61.311540,0.085259,0,...,60.404383,61.230947,62.204662,61.629097,63.277124,0.086179,0.087947,0.089555,0.091254,0.091942


Cambio el formato de los datos para que funcione con el modelo global

In [31]:

cryptos = list(dfs.keys())

df_global = []

for ric, (df, cols) in dfs.items():
    df = df.copy().reset_index()
    df['crypto'] = ric
    df.rename(columns={ric: 'close'}, inplace=True)

    # Renombrar columnas tipo 'BTCUSDT_lag_1' -> 'close_lag_1'
    lag_cols = {f'{ric}_lag_{i}': f'close_lag_{i}' for i in range(1, lags + 1)}
    df.rename(columns=lag_cols, inplace=True)

    df_global.append(df)

# Concatenar todo en un solo DataFrame
df_global = pd.concat(df_global, ignore_index=True)

# Ordenar por fecha
df_global = df_global.sort_values(by='timestamp').reset_index(drop=True)

df_global


,timestamp,close,r,sma,min,max,mom,vol,rsi,atr,...,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5,crypto
0,2020-10-27,13636.17000,0.043770,11554.184333,10542.06000,13636.17000,0.265626,0.017916,77.273397,166.124097,...,74.124078,75.459534,74.446746,75.169194,151.715272,156.141316,158.666879,157.632633,161.501000,BTCUSDT
1,2020-10-27,0.25273,0.018490,0.247623,0.23273,0.25709,0.038588,0.018629,57.099688,0.003870,...,57.840898,59.440599,58.907397,60.127914,0.003843,0.003802,0.003828,0.003908,0.003961,XRPUSDT
2,2020-10-27,4.12170,-0.006361,4.083453,3.44200,4.48060,-0.113308,0.054953,42.733038,0.235048,...,42.892232,43.088247,43.508056,43.960889,0.242246,0.250596,0.258058,0.264381,0.270680,AVAXUSDT
3,2020-10-28,13266.40000,-0.027491,11639.860333,10542.06000,13636.17000,0.240300,0.018860,71.765136,172.912293,...,74.256884,74.124078,75.459534,74.446746,166.124097,151.715272,156.141316,158.666879,157.632633,BTCUSDT
4,2020-10-28,0.24540,-0.029432,0.247767,0.23273,0.25709,0.017962,0.019369,53.598651,0.003985,...,55.317583,57.840898,59.440599,58.907397,0.003870,0.003843,0.003802,0.003828,0.003908,XRPUSDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4558,2024-12-24,98663.58000,0.039087,98381.171000,91965.16000,106133.74000,0.007799,0.025230,57.773555,1730.867671,...,54.791636,57.141430,57.724947,57.443840,1660.135177,1706.870873,1693.117454,1733.795642,1781.734113,BTCUSDT
4559,2024-12-24,41.25000,0.053273,46.232000,36.57000,53.98000,-0.019258,0.065316,51.054260,1.953744,...,46.890348,47.590124,49.646051,48.742458,1.947322,1.926885,1.964018,1.947605,1.979247,AVAXUSDT
4560,2024-12-25,40.25000,-0.024541,46.196667,36.57000,53.98000,-0.025660,0.065387,50.168802,1.921953,...,49.199477,46.890348,47.590124,49.646051,1.953744,1.947322,1.926885,1.964018,1.947605,AVAXUSDT
4561,2024-12-25,99429.60000,0.007734,98595.157333,91965.16000,106133.74000,0.069020,0.023303,58.408279,1698.706082,...,54.456312,54.791636,57.141430,57.724947,1730.867671,1660.135177,1706.870873,1693.117454,1733.795642,BTCUSDT


In [32]:
df_global.columns

Index(['timestamp', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi',
       'atr', 'd', 'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4',
       'close_lag_5', 'r_lag_1', 'r_lag_2', 'r_lag_3', 'r_lag_4', 'r_lag_5',
       'sma_lag_1', 'sma_lag_2', 'sma_lag_3', 'sma_lag_4', 'sma_lag_5',
       'min_lag_1', 'min_lag_2', 'min_lag_3', 'min_lag_4', 'min_lag_5',
       'max_lag_1', 'max_lag_2', 'max_lag_3', 'max_lag_4', 'max_lag_5',
       'mom_lag_1', 'mom_lag_2', 'mom_lag_3', 'mom_lag_4', 'mom_lag_5',
       'vol_lag_1', 'vol_lag_2', 'vol_lag_3', 'vol_lag_4', 'vol_lag_5',
       'rsi_lag_1', 'rsi_lag_2', 'rsi_lag_3', 'rsi_lag_4', 'rsi_lag_5',
       'atr_lag_1', 'atr_lag_2', 'atr_lag_3', 'atr_lag_4', 'atr_lag_5',
       'crypto'],
      dtype='object')

Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [33]:
# Prueba a entrenar sin la columna d_lag_n para ver la importancia que tiene
'''
X_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])
y_train = train['d']
X_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])
y_test = test['d']
'''

"\nX_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])\ny_train = train['d']\nX_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])\ny_test = test['d']\n"

Modelo MLP Classifier GLOBAL

In [25]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
            fin_train = split_date - pd.Timedelta(days=window_pred)
            fin_test= split_date + period
            '''print('split_date, comienzo test', split_date)            
            print('fin_train', fin_train)
            print('fin_test', fin_test )
            print('\n')'''

            if len(test) == 0:
                continue

            drop_cols = ['d', 'timestamp', 'crypto', 'close']
            X_train, y_train = train.drop(columns=drop_cols), train['d']
            X_test, y_test = test.drop(columns=drop_cols), test['d']

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)
            result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")
            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')
        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d', 'timestamp', 'crypto', 'close']
    X_train, y_train = train.drop(columns=drop_cols), train['d']
    X_test, y_test = test.drop(columns=drop_cols), test['d']

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
    sorted_idx = result.importances_mean.argsort()[::-1]
    print("Feature importances (top 10):")
    for i in sorted_idx[:10]:
        print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "Test", frequency=freq)

    return best_params

Cambios hablados 6/5: normalización y eliminación de close_lags_x

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
                ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
                for col in ratio_cols:
                    X[col] = X[col] / close_col
                return X
    
    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
            '''fin_train = split_date - pd.Timedelta(days=window_pred)
            fin_test= split_date + period
            print('split_date, comienzo test', split_date)            
            print('fin_train', fin_train)
            print('fin_test', fin_test )
            print('\n')'''

            if len(test) == 0:
                continue

            drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
            X_train, y_train = train.drop(columns=drop_cols), train['d']
            X_test, y_test = test.drop(columns=drop_cols), test['d']
            
            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train = normalize_with_close(X_train.copy(), close_train)
            X_test = normalize_with_close(X_test.copy(), close_test)

            # timestamp y crypto como índice
            X_train = X_train.set_index(['crypto', 'timestamp'])
            X_test = X_test.set_index(['crypto', 'timestamp'])

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        df_results_test = X_test.copy()
        df_results_test['true'] = y_test.values
        df_results_test['pred'] = pred
        accuracy_per_crypto = df_results_test.groupby(level='crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))

        for crypto, acc_c in accuracy_per_crypto.items():
            crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
    X_train, y_train = train.drop(columns=drop_cols), train['d']
    X_test, y_test = test.drop(columns=drop_cols), test['d']

    # NORMALIZAR CON CLOSE
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train = normalize_with_close(X_train.copy(), close_train)
    X_test = normalize_with_close(X_test.copy(), close_test)

    # timestamp y crypto como índice
    X_train = X_train.set_index(['crypto', 'timestamp'])
    X_test = X_test.set_index(['crypto', 'timestamp'])


    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    # Importancia de cada característica
    '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
    sorted_idx = result.importances_mean.argsort()[::-1]
    print("Feature importances (top 10):")
    for i in sorted_idx[:10]:
        print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "Test", frequency=freq)

    # Accuracy por criptomoneda final
    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    accuracy_per_crypto = df_results_test.groupby(level='crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
    print("\nFINAL TEST - Accuracy por criptomoneda:")
    for crypto, acc_c in accuracy_per_crypto.items():
        print(f"{crypto:<15} | acc = {acc_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "Test", frequency=freq)

    return best_params

Incluimos la columna crypto haciendo un Dummies

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
        ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
        for col in ratio_cols:
            X[col] = X[col] / close_col
        return X

    def prepare_features(df):
        df = df.copy()
        crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
        df = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
        return df, crypto_dummies.columns

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]
            '''fin_train = split_date - pd.Timedelta(days=window_pred)
            fin_test= split_date + period
            print('split_date, comienzo test', split_date)            
            print('fin_train', fin_train)
            print('fin_test', fin_test )
            print('\n')'''

            if len(test) == 0:
                continue

            drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
            X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
            X_test_raw, y_test = test.drop(columns=drop_cols), test['d']

            # Guardar criptos antes de codificar
            cryptos_test = test['crypto'].values

            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
            X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

            # One-hot encoding
            X_train, _ = prepare_features(X_train_raw)
            X_test, _ = prepare_features(X_test_raw)

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            '''result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")'''

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

            df_results_test = X_test.copy()
            df_results_test['true'] = y_test.values
            df_results_test['pred'] = pred
            df_results_test['crypto'] = cryptos_test

            accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
            for crypto, acc_c in accuracy_per_crypto.items():
                crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d'] + [col for col in train.columns if 'close' in col]
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
    cryptos_test = test['crypto'].values

    # Normalizar
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
    X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

    # One-hot encoding
    X_train, _ = prepare_features(X_train_raw)
    X_test, _ = prepare_features(X_test_raw)

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "Test", frequency=freq)

    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    df_results_test['crypto'] = cryptos_test
    accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
    print("\nFINAL TEST - Accuracy por criptomoneda:")
    for crypto, acc_c in accuracy_per_crypto.items():
        print(f"{crypto:<15} | acc = {acc_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "Test", frequency=freq)

    return best_params


In [35]:
      
# Ejecutar la optimización
model_params = {
    "max_iter": 1000,
    "early_stopping": True,
    "validation_fraction": 0.15,
    "shuffle": False,
    "random_state": 100
}

tuned_params = walk_forward_fit_test(MLPClassifier, df_global, frequency, model_params, n_trials=5)


[I 2025-05-07 17:09:44,391] A new study created in memory with name: no-name-2ff7f450-c36e-43a6-b1de-c29d2b1fd5b4


c:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:699: UserWarning: The distribution is specified by [32, 1024] and step=64, but the range is not divisible by `step`. It will be replaced by [32, 992].
  warnings.warn(
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:25: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\3991

VALIDATION | acc=0.7141

Accuracy promedio por criptomoneda (VAL):
AVAXUSDT        | acc = 0.6809
BTCUSDT         | acc = 0.7022
XRPUSDT         | acc = 0.7591


C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:79: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:80: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:79: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:80: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:79: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipyke

VALIDATION | acc=0.7126

Accuracy promedio por criptomoneda (VAL):
AVAXUSDT        | acc = 0.6787
BTCUSDT         | acc = 0.7022
XRPUSDT         | acc = 0.7569


...

XRPUSDT         | acc = 0.7098


[I 2025-05-07 17:10:02,934] Trial 3 finished with value: 0.714074074074074 and parameters: {'hidden_units': 544, 'alpha': 0.000615046468852371, 'learning_rate': 0.006908101171320348}. Best is trial 0 with value: 0.7148148148148148.


VALIDATION | acc=0.7141

Accuracy promedio por criptomoneda (VAL):
AVAXUSDT        | acc = 0.6809
BTCUSDT         | acc = 0.7022
XRPUSDT         | acc = 0.7591
Mejores parámetros encontrados: {'hidden_units': 416, 'alpha': 0.005180756332322513, 'learning_rate': 0.0003234386190564446}


C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:147: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_train = (X_train - mean) / std
C:\Users\raque\AppData\Local\Temp\ipykernel_28756\399147087.py:148: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  X_test = (X_test - mean) / std


FINAL TEST | acc=0.7413

FINAL TEST - Accuracy por criptomoneda:
AVAXUSDT        | acc = 0.6475
BTCUSDT         | acc = 0.7678
XRPUSDT         | acc = 0.8087


Modelo Global

In [20]:
'''def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)

    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        try:
            trial_params = {
                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),
                "max_iter": model_params.get("max_iter", 1000),
                "early_stopping": model_params.get("early_stopping", True),
                "validation_fraction": model_params.get("validation_fraction", 0.15),
                "shuffle": model_params.get("shuffle", False),
                "random_state": model_params.get("random_state", 100),
            }

            results = []
            for split_date in get_split_dates(period, final_test_period):
                global_train, global_test = [], []

                for ric in data:
                    df, cols = dfs[ric]
                    df = df[cols + ['d']]
                    df['timestamp'] = pd.to_datetime(df.index)
                    cutoff = df['timestamp'].max() - final_test_period
                    df_trainval = df[df['timestamp'] < cutoff]

                    train = df_trainval[df_trainval['timestamp'] < split_date]
                    test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

                    if len(test) == 0 or len(train) == 0:
                        continue

                    global_train.append(train)
                    global_test.append(test)

                if not global_train or not global_test:
                    print("[Trial Skipped] No se pudo generar train/test global.")
                    return None

                train_df = pd.concat(global_train)
                test_df = pd.concat(global_test)

                X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
                X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

                mean, std = X_train.mean(), X_train.std()
                std.replace(0, 1, inplace=True)
                X_train = (X_train - mean) / std
                X_test = (X_test - mean) / std

                X_train = X_train.fillna(X_train.mean())
                X_test = X_test.fillna(X_train.mean())

                model = model_class(**trial_params)
                model.fit(X_train, y_train)

                pred = np.where(model.predict(X_test) > 0.5, 1, 0)
                acc = accuracy_score(y_test, pred)
                results.append(acc)

            if not results:
                print("[Trial Skipped] No se generaron métricas.")
                return None

            avg_acc = np.mean(results)
            print(f'[GLOBAL MODEL] acc={avg_acc:.4f}')
            save_results(model_class.__name__, "GLOBAL", acc, "HIPERPARAM-TRAIN")
            return avg_acc

        except Exception as e:
            print(f"[Trial Failed] {e}")
            return None

    def get_split_dates(period, final_test_period):
        all_timestamps = [pd.to_datetime(dfs[ric][0].index) for ric in data]
        min_time = max(min(ts) for ts in all_timestamps)
        max_time = min(max(ts) for ts in all_timestamps)
        cutoff = max_time - final_test_period

        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        return split_dates[-5:]

    # Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # ENTRENAMIENTO FINAL
    global_train, global_test = [], []
    for ric in data:
        df, cols = dfs[ric]
        df = df[cols + ['d']]
        df['timestamp'] = pd.to_datetime(df.index)

        cutoff = df['timestamp'].max() - final_test_period
        train = df[df['timestamp'] < cutoff]
        test = df[df['timestamp'] >= cutoff]

        if len(test) == 0:
            continue

        global_train.append(train)
        global_test.append(test)

    train_df = pd.concat(global_train)
    test_df = pd.concat(global_test)

    X_train, y_train = train_df.drop(columns=['d', 'timestamp']), train_df['d']
    X_test, y_test = test_df.drop(columns=['d', 'timestamp']), test_df['d']

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_train.mean())

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)
    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)

    print(f'[GLOBAL FINAL TEST] acc={acc:.4f}')
    save_results(model_class.__name__, "GLOBAL", acc, "FINAL-TEST")

    return best_params
'''

'def walk_forward_fit_test(model_class, freq, model_params={}, n_trials=5):\n    if freq == \'1h\':\n        period = pd.Timedelta(days=7)\n    elif freq == \'4h\':\n        period = pd.Timedelta(days=15)\n    else:\n        period = pd.Timedelta(days=90)\n\n    final_test_period = pd.Timedelta(days=365)\n\n    def objective(trial):\n        try:\n            trial_params = {\n                "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),\n                "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),\n                "learning_rate_init": trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True),\n                "max_iter": model_params.get("max_iter", 1000),\n                "early_stopping": model_params.get("early_stopping", True),\n                "validation_fraction": model_params.get("validation_fraction", 0.15),\n                "shuffle": model_params.get("shuffle", False),\n                "random_state": model_params.get("rand